In [1]:
!pip install gym==0.26.2

     ---------------------------------------- 0.0/721.7 kB ? eta -:--:--
     --------------------------- ---------- 524.3/721.7 kB 8.5 MB/s eta 0:00:01
     ---------------------------------------- 721.7/721.7 kB 3.3 MB/s  0:00:00
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Created wheel for gym: filename=gym-0.26.2-py3-none-any.whl size=827740 sha256=8d7e91db3918e7bd6327c64ff964d202f2dd406506923a4b7d3c4593855602e3
  Stored in directory: c:\users\nandhini\appdata\local\pip\cache\wheels\af\2b\30\5e78b8b9599f2a2286a582b8da80594f654bf0e18d825a4405
Successfully built gym

   ------------- -------------------------- 1/3 [cloudpickle]
   -------------------------- ------------- 2/3 [gym]
   -----------------

In [2]:
import gym
import numpy as np
import tensorflow as tf
from tensorflow import keras
from collections import deque
import random

# Create environment
env = gym.make("FrozenLake-v1", is_slippery=False)

state_size = env.observation_space.n
action_size = env.action_space.n

# Neural network
model = keras.Sequential([
    keras.layers.Input(shape=(state_size,)),
    keras.layers.Dense(24, activation="relu"),
    keras.layers.Dense(24, activation="relu"),
    keras.layers.Dense(action_size, activation="linear")
])

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="mse"
)

# Parameters
gamma = 0.95
epsilon = 1.0
epsilon_min = 0.01
epsilon_decay = 0.995

memory = deque(maxlen=2000)

# Training
episodes = 500

for episode in range(episodes):

    state, info = env.reset()
    state_vector = np.eye(state_size)[state]
    total_reward = 0

    for step in range(100):

        # Exploration / exploitation
        if random.random() <= epsilon:
            action = env.action_space.sample()
        else:
            q_values = model.predict(
                state_vector.reshape(1, -1), verbose=0
            )
            action = np.argmax(q_values[0])

        next_state, reward, terminated, truncated, info = env.step(action)

        next_vector = np.eye(state_size)[next_state]

        memory.append(
            (state_vector, action, reward, next_vector,
             terminated or truncated)
        )

        state_vector = next_vector
        total_reward += reward

        if terminated or truncated:
            break

    # Train from memory
    if len(memory) >= 32:

        batch = random.sample(memory, 32)

        for state, action, reward, next_state, done in batch:

            target = reward

            if not done:
                next_q = model.predict(
                    next_state.reshape(1, -1), verbose=0
                )
                target += gamma * np.max(next_q[0])

            target_q = model.predict(
                state.reshape(1, -1), verbose=0
            )

            target_q[0][action] = target

            model.fit(
                state.reshape(1, -1),
                target_q,
                epochs=1,
                verbose=0
            )

    if epsilon > epsilon_min:
        epsilon *= epsilon_decay

    if (episode + 1) % 100 == 0:
        print(
            "Episode:", episode + 1,
            "Reward:", total_reward,
            "Epsilon:", round(epsilon, 3)
        )

print("\nTraining completed!")

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
C:\Users\Nandhini\anaconda3\envs\tf\lib\site-packages\requests\__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(

KeyboardInterrupt



In [3]:
!pip install gymnasium


   ---------------------------------------- 0.0/965.4 kB ? eta -:--:--
   ---------- ----------------------------- 262.1/965.4 kB ? eta -:--:--
   ---------------------------------------- 965.4/965.4 kB 5.0 MB/s  0:00:00

   -------------------- ------------------- 1/2 [gymnasium]
   -------------------- ------------------- 1/2 [gymnasium]
   -------------------- ------------------- 1/2 [gymnasium]
   -------------------- ------------------- 1/2 [gymnasium]
   -------------------- ------------------- 1/2 [gymnasium]
   -------------------- ------------------- 1/2 [gymnasium]
   -------------------- ------------------- 1/2 [gymnasium]
   -------------------- ------------------- 1/2 [gymnasium]
   -------------------- ------------------- 1/2 [gymnasium]
   -------------------- ------------------- 1/2 [gymnasium]
   -------------------- ------------------- 1/2 [gymnasium]
   -------------------- ------------------- 1/2 [gymnasium]
   -------------------- ------------------- 1/2 [gymnasium

In [4]:
import gymnasium as gym
import numpy as np
import tensorflow as tf

env = gym.make("FrozenLake-v1", is_slippery=False)

print("TensorFlow version:", tf.__version__)
print("State size:", env.observation_space.n)
print("Action size:", env.action_space.n)


TensorFlow version: 2.10.0
State size: 16
Action size: 4


In [5]:
from tensorflow import keras
import numpy as np

# Create the neural network
model = keras.Sequential([
    keras.layers.Input(shape=(state_size,)),
    keras.layers.Dense(24, activation="relu"),
    keras.layers.Dense(24, activation="relu"),
    keras.layers.Dense(action_size, activation="linear")
])

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="mse"
)

print("RL model created successfully!")
model.summary()

RL model created successfully!
Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense_3 (Dense)             (None, 24)                408       
                                                                 
 dense_4 (Dense)             (None, 24)                600       
                                                                 
 dense_5 (Dense)             (None, 4)                 100       
                                                                 
Total params: 1,108
Trainable params: 1,108
Non-trainable params: 0
_________________________________________________________________


In [6]:
from collections import deque
import random

# Reinforcement Learning parameters
gamma = 0.95
epsilon = 1.0
epsilon_min = 0.01
epsilon_decay = 0.995

memory = deque(maxlen=2000)

episodes = 300

print("Training started...")

for episode in range(episodes):

    state, info = env.reset()
    state_vector = np.eye(state_size)[state]
    total_reward = 0

    for step in range(100):

        # Exploration or exploitation
        if random.random() < epsilon:
            action = env.action_space.sample()
        else:
            q_values = model.predict(
                state_vector.reshape(1, -1),
                verbose=0
            )
            action = np.argmax(q_values[0])

        # Take action
        next_state, reward, terminated, truncated, info = env.step(action)

        next_vector = np.eye(state_size)[next_state]

        done = terminated or truncated

        # Store experience
        memory.append(
            (state_vector, action, reward, next_vector, done)
        )

        state_vector = next_vector
        total_reward += reward

        if done:
            break

    # Train the model
    if len(memory) >= 32:

        batch = random.sample(memory, 32)

        for state, action, reward, next_state, done in batch:

            target = reward

            if not done:
                next_q = model.predict(
                    next_state.reshape(1, -1),
                    verbose=0
                )
                target += gamma * np.max(next_q[0])

            target_q = model.predict(
                state.reshape(1, -1),
                verbose=0
            )

            target_q[0][action] = target

            model.fit(
                state.reshape(1, -1),
                target_q,
                epochs=1,
                verbose=0
            )

    # Reduce exploration
    if epsilon > epsilon_min:
        epsilon *= epsilon_decay

    if (episode + 1) % 50 == 0:
        print(
            "Episode:", episode + 1,
            "Reward:", total_reward,
            "Epsilon:", round(epsilon, 3)
        )

print("\nTraining completed!")

Training started...



KeyboardInterrupt



In [7]:
from collections import deque
import random
import numpy as np

gamma = 0.95
epsilon = 1.0
epsilon_min = 0.01
epsilon_decay = 0.99

memory = deque(maxlen=2000)

episodes = 100
batch_size = 32

print("Training started...")

for episode in range(episodes):

    state, info = env.reset()
    total_reward = 0

    for step in range(50):

        state_vector = np.eye(state_size)[state]

        # Exploration / exploitation
        if random.random() < epsilon:
            action = env.action_space.sample()
        else:
            q_values = model.predict(
                state_vector.reshape(1, -1),
                verbose=0
            )
            action = np.argmax(q_values[0])

        next_state, reward, terminated, truncated, info = env.step(action)

        next_vector = np.eye(state_size)[next_state]
        done = terminated or truncated

        memory.append(
            (state_vector, action, reward, next_vector, done)
        )

        state = next_state
        total_reward += reward

        if done:
            break

    # Train using one batch
    if len(memory) >= batch_size:

        batch = random.sample(memory, batch_size)

        states = np.array([x[0] for x in batch])
        next_states = np.array([x[3] for x in batch])

        q_values = model.predict(states, verbose=0)
        next_q_values = model.predict(next_states, verbose=0)

        for i, (_, action, reward, _, done) in enumerate(batch):

            if done:
                q_values[i][action] = reward
            else:
                q_values[i][action] = (
                    reward + gamma * np.max(next_q_values[i])
                )

        model.fit(
            states,
            q_values,
            epochs=1,
            verbose=0
        )

    if epsilon > epsilon_min:
        epsilon *= epsilon_decay

    if (episode + 1) % 20 == 0:
        print(
            "Episode:", episode + 1,
            "Reward:", total_reward,
            "Epsilon:", round(epsilon, 3)
        )

print("\nTraining completed!")

Training started...
Episode: 20 Reward: 0.0 Epsilon: 0.818
Episode: 40 Reward: 0.0 Epsilon: 0.669
Episode: 60 Reward: 0.0 Epsilon: 0.547
Episode: 80 Reward: 1.0 Epsilon: 0.448
Episode: 100 Reward: 1.0 Epsilon: 0.366

Training completed!


In [8]:
# Evaluate the trained robot

state, info = env.reset()
total_reward = 0

print("Robot Navigation Evaluation")
print("---------------------------")

for step in range(20):

    state_vector = np.eye(state_size)[state]

    # Choose the best learned action
    q_values = model.predict(
        state_vector.reshape(1, -1),
        verbose=0
    )

    action = np.argmax(q_values[0])

    next_state, reward, terminated, truncated, info = env.step(action)

    print(
        "Step:", step + 1,
        "| State:", state,
        "| Action:", action,
        "| Reward:", reward
    )

    total_reward += reward
    state = next_state

    if terminated or truncated:
        break

print("---------------------------")
print("Final State:", state)
print("Total Reward:", total_reward)

if total_reward > 0:
    print("Robot reached the goal successfully!")
else:
    print("Robot did not reach the goal.")

Robot Navigation Evaluation
---------------------------
Step: 1 | State: 0 | Action: 0 | Reward: 0.0
Step: 2 | State: 0 | Action: 0 | Reward: 0.0
Step: 3 | State: 0 | Action: 0 | Reward: 0.0
Step: 4 | State: 0 | Action: 0 | Reward: 0.0
Step: 5 | State: 0 | Action: 0 | Reward: 0.0
Step: 6 | State: 0 | Action: 0 | Reward: 0.0
Step: 7 | State: 0 | Action: 0 | Reward: 0.0
Step: 8 | State: 0 | Action: 0 | Reward: 0.0
Step: 9 | State: 0 | Action: 0 | Reward: 0.0
Step: 10 | State: 0 | Action: 0 | Reward: 0.0
Step: 11 | State: 0 | Action: 0 | Reward: 0.0
Step: 12 | State: 0 | Action: 0 | Reward: 0.0
Step: 13 | State: 0 | Action: 0 | Reward: 0.0
Step: 14 | State: 0 | Action: 0 | Reward: 0.0
Step: 15 | State: 0 | Action: 0 | Reward: 0.0
Step: 16 | State: 0 | Action: 0 | Reward: 0.0
Step: 17 | State: 0 | Action: 0 | Reward: 0.0
Step: 18 | State: 0 | Action: 0 | Reward: 0.0
Step: 19 | State: 0 | Action: 0 | Reward: 0.0
Step: 20 | State: 0 | Action: 0 | Reward: 0.0
---------------------------
Final

In [9]:
# Train the robot again with more episodes

episodes = 500
epsilon = 1.0

print("Training started again...")

for episode in range(episodes):

    state, info = env.reset()
    total_reward = 0

    for step in range(50):

        state_vector = np.eye(state_size)[state]

        if random.random() < epsilon:
            action = env.action_space.sample()
        else:
            q_values = model.predict(
                state_vector.reshape(1, -1),
                verbose=0
            )
            action = np.argmax(q_values[0])

        next_state, reward, terminated, truncated, info = env.step(action)

        next_vector = np.eye(state_size)[next_state]
        done = terminated or truncated

        memory.append(
            (state_vector, action, reward, next_vector, done)
        )

        state = next_state
        total_reward += reward

        if done:
            break

    if len(memory) >= 32:

        batch = random.sample(memory, 32)

        states = np.array([x[0] for x in batch])
        next_states = np.array([x[3] for x in batch])

        q_values = model.predict(states, verbose=0)
        next_q_values = model.predict(next_states, verbose=0)

        for i, (_, action, reward, _, done) in enumerate(batch):

            if done:
                q_values[i][action] = reward
            else:
                q_values[i][action] = (
                    reward + gamma * np.max(next_q_values[i])
                )

        model.fit(
            states,
            q_values,
            epochs=1,
            verbose=0
        )

    if epsilon > epsilon_min:
        epsilon *= epsilon_decay

    if (episode + 1) % 100 == 0:
        print(
            "Episode:", episode + 1,
            "Reward:", total_reward,
            "Epsilon:", round(epsilon, 3)
        )

print("Training completed again!")

Training started again...
Episode: 100 Reward: 1.0 Epsilon: 0.366
Episode: 200 Reward: 0.0 Epsilon: 0.134
Episode: 300 Reward: 1.0 Epsilon: 0.049
Episode: 400 Reward: 1.0 Epsilon: 0.018
Episode: 500 Reward: 1.0 Epsilon: 0.01
Training completed again!
